Download the libraries

In [ ]:
#%pip install ultralytics
#%pip install torch torchvision torchaudio
#%pip install imgaug
#%pip install -U segmentation-models
#%pip install mrcnn
#%pip install tensorflow_model_optimization
#%pip install tensorflow==1.15.0
#%pip install keras==2.2.5
#%pip install numpy==1.19.5
#%pip install h5py==2.10.0#
#%pip install scikit-image pillow-6.2.2 cython matplotlib
#%pip install Pillow
#%pip install numpy
#%pip install matplotlib
#%pip install opencv-python
#%pip install opencv-contrib-python
#%pip install scikit-image
#%pip install webcolors
#%pip install scikit-image pillow matplotlib Cython
# 1. Clear out any bad PyTorch installations entirely
#%pip uninstall torch torchvision torchaudio -y
# 2. Clear pip's internal cache so it doesn't try to reuse the CPU file
#%pip cache purge
# 3. Force-install the dedicated CUDA 12.4 version by using the --no-cache-dir flag
#%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124 --no-cache-dir

In [34]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
import os

Create Manually the polyline and save in a image the mask

In [ ]:
%matplotlib widget
# 1. Load your image to get the correct canvas dimensions
image_path =  r"..\images\val\0100.jpg"
try:
    img = cv2.imread(image_path)
    h, w, c = img.shape
    # Convert for display
    img_display = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
except Exception as e:
    print(f"Could not load image, using fallback size. Error: {e}")
    h, w, c = 500, 500, 3
    img_display = np.ones((h, w, c), dtype=np.uint8) * 128

# List to store our clicked points
points = []

# 2. Set up the interactive plot
fig, ax = plt.subplots(figsize=(20, 8))
ax.set_title("")
im_plot = ax.imshow(img_display)

# 3. Mouse click handler
def onclick(event):
    if event.xdata is None or event.ydata is None:
        return
    
    ix, iy = int(event.xdata), int(event.ydata)
    points.append((ix, iy))
    
    # Draw point and line on the interactive display
    ax.plot(ix, iy, 'ro', markersize=5) 
    if len(points) > 1:
        ax.plot([points[-2][0], points[-1][0]], [points[-2][1], points[-1][1]], 'b-', linewidth=2)
        
    fig.canvas.draw()

# Connect mouse click event
cid_click = fig.canvas.mpl_connect('button_press_event', onclick)

# 4. Create an interactive Jupyter Button to save the mask
btn_save = widgets.Button(
    description="Save Polyline Mask",
    button_style="success", # Makes the button green
    icon="check",
    layout=widgets.Layout(width='200px', height='40px')
)

# Output area to print success messages below the button
out = widgets.Output()

def on_button_clicked(b):
    with out:
        if len(points) < 2:
            print("You need at least 2 points to draw a polyline!")
            return
            
        # Create a completely black canvas matching the original image size
        mask = np.zeros((h, w), dtype=np.uint8)
        
        # Format points for OpenCV
        pts = np.array(points, np.int32).reshape((-1, 1, 2))
        
        # Draw ONLY the polyline (white line on black background)
        cv2.polylines(mask, [pts], isClosed=False, color=255, thickness=3)
        
        # Export the polyline-only mask
        output_path = image_path + ".png"
        cv2.imwrite(output_path, mask)
        
        print(f"[SUCCESS] Polyline-only mask saved to: {output_path}")
        
        # Disconnect click listener and close plot to clean up memory
        fig.canvas.mpl_disconnect(cid_click)
        plt.close(fig)

# Bind the save function to the button click
btn_save.on_click(on_button_clicked)

# Display the plot, the button, and the output console
plt.show()
display(btn_save, out)

Create Lables using the mask

In [ ]:

# Paste your copied paths here:
input_dir = r'..\images\train\mask train'
output_dir =  r'..\images\labels\train'
os.makedirs(output_dir, exist_ok=True)

In [4]:
for j in os.listdir(input_dir):
    # Skip any system hidden files like .DS_Store
    if not j.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
        continue
        
    image_path = os.path.join(input_dir, j)
    mask = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        continue

    _, mask = cv2.threshold(mask, 1, 255, cv2.THRESH_BINARY)
    H, W = mask.shape
    
    # --- FIX 1: Change to CHAIN_APPROX_NONE to keep all point coordinates ---
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)

    polygons = []
    for cnt in contours:
        # --- FIX 2: Filter by curve length (perimeter) instead of area ---
        # Polylines have almost 0 area, but they have physical length (perimeter)
        perimeter = cv2.arcLength(cnt, True)
        
        if perimeter > 50:  # Adjust this threshold to filter out tiny noise dots
            
            # --- FIX 3: Downsample points to a maximum of 100 points (200 values) ---
            num_points = len(cnt)
            if num_points > 100:
                # Mathematically select 100 perfectly even steps along the path
                indices = np.linspace(0, num_points - 1, 100, dtype=int)
                sampled_cnt = cnt[indices]
            else:
                sampled_cnt = cnt

            polygon = []
            for point in sampled_cnt:
                x, y = point[0]
                polygon.append(x / W)
                polygon.append(y / H)
            polygons.append(polygon)

    # --- FILE WRITING BLOCK ---
    base_name = os.path.splitext(j)[0]
    txt_path = os.path.join(output_dir, f"{base_name}.txt")
    
    with open(txt_path, 'w') as f:
        for polygon in polygons:
            # Map every float coordinate to a 6-decimal string
            str_coords = [f"{p:.6f}" for p in polygon]
            # Write class 0 followed by coordinates
            f.write(f"0 {' '.join(str_coords)}\n")

Configure my video card for use CUDA

In [5]:
import torch
print("CUDA Available:", torch.cuda.is_available())
print("GPU Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

CUDA Available: True
GPU Name: NVIDIA GeForce RTX 3070


Configure the Video Card and run the model

In [ ]:
from ultralytics import YOLO
# 1. Load the model
model = YOLO('yolo11n-seg.pt')
config_path = r'../config.yaml'

# 2. Train with a standard minimum image size (must be multiple of 32)
# Using 'rect=True' helps if your images aren't square
results = model.train(
    data=config_path,
    epochs=50, 
    imgsz=640, 
    device=0,        # Keeps running on your RTX 3070
    workers=0,       # <--- CRUCIAL FIX 1: Disables Windows multi-threading data time-outs
    batch=4,         # <--- CRUCIAL FIX 2: Sets a safe, lightweight initial batch size
    plots=True
)


New https://pypi.org/project/ultralytics/8.4.117 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.36  Python-3.13.9 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 3070, 8192MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\Datapixer\Potholes Images\config.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0

Export and analize the results

In [ ]:
metrics = model.val(
    workers=0,   # <--- THIS STOPS THE VALIDATION CRASH
    batch=4      # <--- Keeps memory usage light and stable
)

# 3. Print the metrics
print(f"Mean Average Precision (Mask mAP50-95): {metrics.seg.map:.4f}")
print(f"Mean Average Precision (Mask mAP50): {metrics.seg.map50:.4f}")

In [ ]:
import pandas as pd
# Load metadata from the Excel file
df_images = pd.read_excel('../Data.xlsx', sheet_name='DATA')

# Get unique combinations of Category, Subcategory, and Urgency
unique_combos = (
    df_images[['CATEGORY', 'SUB_CATEGORIES', 'URGENCY']]
    .drop_duplicates()
    .reset_index(drop=True)
)

# Build the YAML names dictionary
yaml_lines = [
    "path:  ../Potholes Images",
    "train: images/train",
    "val: images/val",
    "names:",
]

for idx, row in unique_combos.iterrows():
    # Replace spaces and special characters with safe identifiers for class naming
    cat = row['CATEGORY'].replace(' / ', '_').replace(' ', '_')
    sub = row['SUB_CATEGORIES'].replace(' / ', '_').replace(' ', '_')
    urg = row['URGENCY']
    compound_name = f"{cat}__{sub}__{urg}"
    yaml_lines.append(f"  {idx}: {compound_name}")

# Save to config.yaml
with open("config.yaml", "w") as f:
    f.write("\n".join(yaml_lines))

print(f"Successfully generated config.yaml with {len(unique_combos)} classes!")

Successfully generated config.yaml with 33 classes!


In [ ]:
from ultralytics import YOLO
# Load your trained weights
model = YOLO(r'..\segment\train\weights\best.pt')

In [ ]:
# Run prediction on a test image
results = model.predict(
    source=r'..\images\val\0101.jpg', 
    conf=0.1,    # You can adjust this to see more or fewer detections
    save=True,   # This saves the annotated image automatically
    show=False   # Set to True if you want it to pop up in a new window
)

for r in results:
  for box in r.boxes:
    class_id = int(box.cls[0])
    confidence = float(box.conf[0])
    full_label = str(model.names[class_id])
    # Check if the label contains our custom compound delimiter '__'
    if '__' in full_label:
      cat_raw, sub_raw, urgency = full_label.split('__')
      category = cat_raw.replace('_', ' ').replace('&', '&')
      subcategory = sub_raw.replace('_', ' ')
    else:
      # Fallback lookup directly from our unique_combos table using class_id index
      if class_id < len(unique_combos):
        row = unique_combos.iloc[class_id]
        category = row['CATEGORY']
        subcategory = row['SUB_CATEGORIES']
        urgency = row['URGENCY']
      else:
        category = 'Unknown'
        subcategory = 'Unknown'
        urgency = 'Unknown'

    print('=== Road Distress Analysis Results ===')
    print(f'• Recognition Mask: Detected (Confidence: {confidence:.2f})')
    print(f'• Category:         {category}')
    print(f'• Sub-Category:     {subcategory}')
    print(f'• Urgency Level:    {urgency}')

In [ ]:
results = model.predict(
    source=r'..\images\val\0102.jpg', 
    conf=0.1,    # You can adjust this to see more or fewer detections
    save=True,   # This saves the annotated image automatically
    show=False   # Set to True if you want it to pop up in a new window
)

for r in results:
  for box in r.boxes:
    class_id = int(box.cls[0])
    confidence = float(box.conf[0])
    full_label = str(model.names[class_id])
    # Check if the label contains our custom compound delimiter '__'
    if '__' in full_label:
      cat_raw, sub_raw, urgency = full_label.split('__')
      category = cat_raw.replace('_', ' ').replace('&', '&')
      subcategory = sub_raw.replace('_', ' ')
    else:
      # Fallback lookup directly from our unique_combos table using class_id index
      if class_id < len(unique_combos):
        row = unique_combos.iloc[class_id]
        category = row['CATEGORY']
        subcategory = row['SUB_CATEGORIES']
        urgency = row['URGENCY']
      else:
        category = 'Unknown'
        subcategory = 'Unknown'
        urgency = 'Unknown'

    print('=== Road Distress Analysis Results ===')
    print(f'• Recognition Mask: Detected (Confidence: {confidence:.2f})')
    print(f'• Category:         {category}')
    print(f'• Sub-Category:     {subcategory}')
    print(f'• Urgency Level:    {urgency}')

In [ ]:
results = model.predict(
    source=r'..\images\val\0103.jpg', 
    conf=0.1,    # You can adjust this to see more or fewer detections
    save=True,   # This saves the annotated image automatically
    show=False   # Set to True if you want it to pop up in a new window
)

for r in results:
  for box in r.boxes:
    class_id = int(box.cls[0])
    confidence = float(box.conf[0])
    full_label = str(model.names[class_id])
    # Check if the label contains our custom compound delimiter '__'
    if '__' in full_label:
      cat_raw, sub_raw, urgency = full_label.split('__')
      category = cat_raw.replace('_', ' ').replace('&', '&')
      subcategory = sub_raw.replace('_', ' ')
    else:
      # Fallback lookup directly from our unique_combos table using class_id index
      if class_id < len(unique_combos):
        row = unique_combos.iloc[class_id]
        category = row['CATEGORY']
        subcategory = row['SUB_CATEGORIES']
        urgency = row['URGENCY']
      else:
        category = 'Unknown'
        subcategory = 'Unknown'
        urgency = 'Unknown'

    print('=== Road Distress Analysis Results ===')
    print(f'• Recognition Mask: Detected (Confidence: {confidence:.2f})')
    print(f'• Category:         {category}')
    print(f'• Sub-Category:     {subcategory}')
    print(f'• Urgency Level:    {urgency}')

In [ ]:
results = model.predict(
    source=r'..\images\val\0104.jpg', 
    conf=0.1,    # You can adjust this to see more or fewer detections
    save=True,   # This saves the annotated image automatically
    show=False   # Set to True if you want it to pop up in a new window
)

for r in results:
  for box in r.boxes:
    class_id = int(box.cls[0])
    confidence = float(box.conf[0])
    full_label = str(model.names[class_id])
    # Check if the label contains our custom compound delimiter '__'
    if '__' in full_label:
      cat_raw, sub_raw, urgency = full_label.split('__')
      category = cat_raw.replace('_', ' ').replace('&', '&')
      subcategory = sub_raw.replace('_', ' ')
    else:
      # Fallback lookup directly from our unique_combos table using class_id index
      if class_id < len(unique_combos):
        row = unique_combos.iloc[class_id]
        category = row['CATEGORY']
        subcategory = row['SUB_CATEGORIES']
        urgency = row['URGENCY']
      else:
        category = 'Unknown'
        subcategory = 'Unknown'
        urgency = 'Unknown'

    print('=== Road Distress Analysis Results ===')
    print(f'• Recognition Mask: Detected (Confidence: {confidence:.2f})')
    print(f'• Category:         {category}')
    print(f'• Sub-Category:     {subcategory}')
    print(f'• Urgency Level:    {urgency}')